<a href="https://colab.research.google.com/github/serdararici/btk-datathon-2026/blob/main/notebooks/v4_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Datathon 2026 — v4: Ensemble + Hyperparameter Tuning (FINAL)
**Goal:** Best possible model — ensemble of XGBoost, LightGBM, CatBoost  
**Previous:** v3 → Kaggle MSE 97.33  
**Author:** Serdar Arıcı | **Date:** June 2026

In [1]:
# ============================================================
# SECTION 0: INSTALL LIBRARIES
# ============================================================

# CatBoost and Optuna are not pre-installed in Colab
!pip install catboost optuna -q

print("Libraries installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 20.7 MB/s eta 0:00:00
Libraries installed!


In [2]:
# ============================================================
# SECTION 1: SETUP
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

DATA_PATH    = '/content/drive/MyDrive/Colab Notebooks/btk-datathon-2026/'
DATASET_PATH = DATA_PATH + 'datasets/'
OUTPUT_PATH  = DATA_PATH + 'outputs/'

train = pd.read_csv(DATASET_PATH + 'train.csv', encoding='utf-8-sig')
test  = pd.read_csv(DATASET_PATH + 'test_x.csv', encoding='utf-8-sig')

print(f"Train: {train.shape}")
print(f"Test:  {test.shape}")

Mounted at /content/drive
Train: (10000, 47)
Test:  (10000, 46)


In [3]:
# ============================================================
# SECTION 2: FEATURE ENGINEERING + NLP + PREPROCESSING
# ============================================================

import re
from sklearn.feature_extraction.text import TfidfVectorizer

# ---- STEP 1: NUMERIC FEATURE ENGINEERING ----
train_fe = train.copy()
test_fe  = test.copy()

def create_features(df):
    technical_cols = ['coding_score', 'problem_solving_score', 'data_structures_score',
                      'sql_score', 'machine_learning_score', 'backend_score',
                      'frontend_score', 'cloud_score', 'devops_score']
    df['avg_technical_score'] = df[technical_cols].mean(axis=1)

    soft_cols = ['communication_score', 'teamwork_score',
                 'leadership_score', 'presentation_score']
    df['avg_soft_skill_score'] = df[soft_cols].mean(axis=1)

    df['experience_score'] = (
        df['internship_count'] * 3 + df['real_client_project_count'] * 2 +
        df['freelance_project_count'] * 1 + df['hackathon_count'] * 1
    )
    df['interview_success_rate'] = df['interviews_attended'] / (df['applications_sent'] + 1)
    df['hackathon_efficiency']   = df['hackathon_awards'] / (df['hackathon_count'] + 1)
    df['github_impact']          = df['github_repo_count'] * df['github_avg_stars'].fillna(0)
    df['total_project_count']    = df['real_client_project_count'] + df['freelance_project_count']
    return df

train_fe = create_features(train_fe)
test_fe  = create_features(test_fe)

# Role weighted score
def create_role_weighted_score(df):
    role_weights = {
        'Data Scientist':       {'machine_learning_score': 3, 'problem_solving_score': 2, 'sql_score': 2, 'data_structures_score': 1, 'coding_score': 1},
        'Data Analyst':         {'sql_score': 3, 'problem_solving_score': 2, 'machine_learning_score': 1, 'coding_score': 1, 'frontend_score': 1},
        'Backend Developer':    {'backend_score': 3, 'coding_score': 2, 'data_structures_score': 2, 'sql_score': 1, 'devops_score': 1},
        'Frontend Developer':   {'frontend_score': 3, 'coding_score': 2, 'problem_solving_score': 1, 'backend_score': 1, 'data_structures_score': 1},
        'AI Engineer':          {'machine_learning_score': 3, 'coding_score': 2, 'data_structures_score': 2, 'cloud_score': 1, 'problem_solving_score': 1},
        'MLOps Engineer':       {'devops_score': 3, 'machine_learning_score': 2, 'cloud_score': 2, 'backend_score': 1, 'coding_score': 1},
        'Cloud Engineer':       {'cloud_score': 3, 'devops_score': 2, 'backend_score': 2, 'coding_score': 1, 'problem_solving_score': 1},
        'DevOps Engineer':      {'devops_score': 3, 'cloud_score': 2, 'backend_score': 2, 'coding_score': 1, 'problem_solving_score': 1},
        'Cybersecurity Analyst':{'problem_solving_score': 3, 'coding_score': 2, 'backend_score': 2, 'devops_score': 1, 'data_structures_score': 1},
        'Product Analyst':      {'problem_solving_score': 3, 'sql_score': 2, 'machine_learning_score': 1, 'communication_score': 1, 'frontend_score': 1},
        'Software Developer':   {'coding_score': 3, 'data_structures_score': 2, 'backend_score': 2, 'problem_solving_score': 1, 'frontend_score': 1},
    }
    weighted_scores = []
    for _, row in df.iterrows():
        role = row['target_role']
        if role in role_weights:
            weights    = role_weights[role]
            total_w    = sum(weights.values())
            score      = sum(row[skill] * w for skill, w in weights.items()) / total_w
        else:
            score = row['avg_technical_score']
        weighted_scores.append(score)
    df['role_weighted_score'] = weighted_scores
    return df

train_fe = create_role_weighted_score(train_fe)
test_fe  = create_role_weighted_score(test_fe)

print("Numeric feature engineering done!")

# ---- STEP 2: NLP FEATURES ----
def clean_text(text):
    if pd.isnull(text): return ""
    text = text.replace('İ', 'i').replace('I', 'ı')
    text = text.lower()
    text = re.sub(r'[^a-zçğıöşü\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

train_fe['feedback_clean'] = train_fe['mentor_feedback_text'].apply(clean_text)
test_fe['feedback_clean']  = test_fe['mentor_feedback_text'].apply(clean_text)

positive_words = [
    'dikkat', 'çekici', 'güçlü', 'başarı', 'başarılı', 'yetkin', 'yetenek',
    'yetenekli', 'uzmanlık', 'mükemmel', 'kanıtlıyor', 'sağlam', 'ileri',
    'kayda', 'değer', 'avantaj', 'aranan', 'kaliteli', 'üstün', 'parlak',
    'etkileyici', 'olumlu', 'güzel', 'yüksek', 'kuvvetli', 'önemli'
]
negative_words = [
    'ancak', 'gereken', 'gerekiyor', 'gerekmekte', 'gerekebilir', 'gerekli',
    'eksik', 'zayıf', 'yetersiz', 'geliştirmesi', 'geliştirmek', 'çalışması',
    'destek', 'gelişime', 'gelişmekte', 'sorun', 'düşük', 'daha', 'fazla'
]
potential_words = [
    'potansiyel', 'potansiyeli', 'umut', 'verici', 'motivasyon',
    'gelişim', 'çaba', 'azim', 'azmi', 'hedef', 'hedefleri'
]

def count_words(text, word_list):
    words = text.split()
    return sum(1 for w in words if w in word_list)

def create_sentiment_features(df):
    df['positive_count'] = df['feedback_clean'].apply(lambda x: count_words(x, positive_words))
    df['negative_count'] = df['feedback_clean'].apply(lambda x: count_words(x, negative_words))
    df['potential_count'] = df['feedback_clean'].apply(lambda x: count_words(x, potential_words))
    df['sentiment_score'] = df['positive_count'] - df['negative_count']
    df['sentiment_ratio'] = df['positive_count'] / (df['positive_count'] + df['negative_count'] + 1)
    return df

train_fe = create_sentiment_features(train_fe)
test_fe  = create_sentiment_features(test_fe)

# TF-IDF
turkish_stopwords = [
    've', 'ile', 'bir', 'bu', 'da', 'de', 'için', 'olan', 'olarak',
    'çok', 'gibi', 'kadar', 'sonra', 'daha', 'en', 'ya', 'veya',
    'ama', 'ise', 'göre', 'her', 'ki', 'mi', 'ne', 'o', 'şu'
]

tfidf = TfidfVectorizer(max_features=50, stop_words=turkish_stopwords, ngram_range=(1, 2))
tfidf_train = tfidf.fit_transform(train_fe['feedback_clean'])
tfidf_test  = tfidf.transform(test_fe['feedback_clean'])

tfidf_cols      = [f'tfidf_{w}' for w in tfidf.get_feature_names_out()]
tfidf_train_df  = pd.DataFrame(tfidf_train.toarray(), columns=tfidf_cols, index=train_fe.index)
tfidf_test_df   = pd.DataFrame(tfidf_test.toarray(),  columns=tfidf_cols, index=test_fe.index)

print("NLP features done!")

# ---- STEP 3: PREPROCESSING ----
train_processed = train_fe.copy()
test_processed  = test_fe.copy()

# Imputation
duration_median = train_processed.loc[train_processed['internship_count'] > 0, 'internship_duration_months'].median()
for df in [train_processed, test_processed]:
    df.loc[(df['internship_count'] == 0) & (df['internship_duration_months'].isnull()), 'internship_duration_months'] = 0
    df.loc[(df['internship_count'] > 0)  & (df['internship_duration_months'].isnull()), 'internship_duration_months'] = duration_median

numeric_to_impute = ['english_exam_score', 'portfolio_score', 'github_avg_stars',
                     'open_source_contribution_count', 'linkedin_profile_score', 'hr_interview_score']
medians = {col: train_processed[col].median() for col in numeric_to_impute}
train_processed.fillna(medians, inplace=True)
test_processed.fillna(medians, inplace=True)

# Encoding
tier_mapping = {'Tier 1': 4, 'Tier 2': 3, 'Tier 3': 2, 'Tier 4': 1}
train_processed['university_tier'] = train_processed['university_tier'].map(tier_mapping)
test_processed['university_tier']  = test_processed['university_tier'].map(tier_mapping)

target_col      = train_processed['career_success_score'].copy()
train_processed = train_processed.drop(columns=['career_success_score'])

ohe_cols        = ['department', 'target_role']
train_processed = pd.get_dummies(train_processed, columns=ohe_cols, dtype=int)
test_processed  = pd.get_dummies(test_processed,  columns=ohe_cols, dtype=int)

train_processed, test_processed = train_processed.align(test_processed, join='left', axis=1, fill_value=0)

noise_cols = ['hobby', 'preferred_social_media_platform', 'student_id',
              'mentor_feedback_text', 'feedback_clean']
for col in noise_cols:
    if col in train_processed.columns: train_processed = train_processed.drop(columns=[col])
    if col in test_processed.columns:  test_processed  = test_processed.drop(columns=[col])

# Combine with NLP
train_processed = train_processed.reset_index(drop=True)
test_processed  = test_processed.reset_index(drop=True)
tfidf_train_df  = tfidf_train_df.reset_index(drop=True)
tfidf_test_df   = tfidf_test_df.reset_index(drop=True)

train_final = pd.concat([train_processed, tfidf_train_df], axis=1)
test_final  = pd.concat([test_processed,  tfidf_test_df],  axis=1)

# Remove duplicates, add target back
train_final = train_final.loc[:, ~train_final.columns.duplicated()]
test_final  = test_final.loc[:, ~test_final.columns.duplicated()]

train_final = train_final.astype(float)
test_final  = test_final.astype(float)

train_final['career_success_score'] = target_col.reset_index(drop=True)

print(f"\nFinal shape - Train: {train_final.shape}")
print(f"Final shape - Test:  {test_final.shape}")
print(f"Missing train: {train_final.isnull().sum().sum()}")
print(f"Missing test:  {test_final.isnull().sum().sum()}")

Numeric feature engineering done!
NLP features done!

Final shape - Train: (10000, 122)
Final shape - Test:  (10000, 121)
Missing train: 0
Missing test:  0


In [4]:
# ============================================================
# SECTION 3: PREPARE X AND y
# ============================================================

from sklearn.model_selection import KFold

# Separate features and target
X = train_final.drop(columns=['career_success_score']).to_numpy(dtype=float)
y = train_final['career_success_score'].to_numpy(dtype=float)
X_test = test_final.to_numpy(dtype=float)

# Feature names (for importance analysis later)
feature_names = train_final.drop(columns=['career_success_score']).columns.tolist()

# 5-Fold Cross Validation setup
N_FOLDS = 5
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

print(f"X shape:      {X.shape}")
print(f"y shape:      {y.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"CV folds:     {N_FOLDS}")

X shape:      (10000, 121)
y shape:      (10000,)
X_test shape: (10000, 121)
CV folds:     5


In [8]:
# ============================================================
# SECTION 4: HYPERPARAMETER TUNING - XGBOOST
# ============================================================

import optuna
from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_score

optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective_xgb(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 200, 600),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth':        trial.suggest_int('max_depth', 3, 7),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'random_state': 42,
        'n_jobs': 1
    }
    model = XGBRegressor(**params)
    scores = cross_val_score(model, X, y, cv=3,
                             scoring='neg_mean_squared_error', n_jobs=-1)
    return -scores.mean()

print("Starting XGBoost tuning (20 trials, 3-fold CV)...")

study_xgb = optuna.create_study(direction='minimize')
study_xgb.optimize(objective_xgb, n_trials=20)

best_params_xgb = study_xgb.best_params
best_mse_xgb    = study_xgb.best_value

print(f"Best XGBoost MSE: {best_mse_xgb:.4f}")
print(f"Best parameters:  {best_params_xgb}")

Starting XGBoost tuning (20 trials, 3-fold CV)...
Best XGBoost MSE: 81.9021
Best parameters:  {'n_estimators': 538, 'learning_rate': 0.01979205419151862, 'max_depth': 6, 'subsample': 0.6354143952235325, 'colsample_bytree': 0.8578660758455927}


In [9]:
# ============================================================
# SECTION 5: HYPERPARAMETER TUNING WITH OPTUNA (LightGBM)
# ============================================================

import lightgbm as lgb

def objective_lgb(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 200, 600),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth':        trial.suggest_int('max_depth', 3, 7),
        'num_leaves':       trial.suggest_int('num_leaves', 20, 100),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'random_state': 42,
        'n_jobs': 1,
        'verbose': -1        # suppress LightGBM logs
    }

    model = lgb.LGBMRegressor(**params)

    scores = cross_val_score(
        model, X, y,
        cv=3,
        scoring='neg_mean_squared_error',
        n_jobs=-1
    )

    return -scores.mean()

print("Starting LightGBM tuning (20 trials, 3-fold CV)...")

study_lgb = optuna.create_study(direction='minimize')
study_lgb.optimize(objective_lgb, n_trials=20)

best_params_lgb = study_lgb.best_params
best_mse_lgb    = study_lgb.best_value

print(f"Best LightGBM MSE: {best_mse_lgb:.4f}")
print(f"Best parameters:   {best_params_lgb}")

Starting LightGBM tuning (20 trials, 3-fold CV)...
Best LightGBM MSE: 81.7304
Best parameters:   {'n_estimators': 499, 'learning_rate': 0.04113818489322525, 'max_depth': 6, 'num_leaves': 20, 'subsample': 0.6089010419351871, 'colsample_bytree': 0.6032402052884556}


In [10]:
# ============================================================
# SECTION 6: HYPERPARAMETER TUNING - CATBOOST
# ============================================================

from catboost import CatBoostRegressor

def objective_cat(trial):
    params = {
        'iterations':    trial.suggest_int('iterations', 200, 600),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'depth':         trial.suggest_int('depth', 3, 8),
        'l2_leaf_reg':   trial.suggest_float('l2_leaf_reg', 1, 10),
        'random_state': 42,
        'verbose': 0
    }
    model = CatBoostRegressor(**params)
    scores = cross_val_score(model, X, y, cv=3,
                             scoring='neg_mean_squared_error', n_jobs=-1)
    return -scores.mean()

print("Starting CatBoost tuning (20 trials, 3-fold CV)...")

study_cat = optuna.create_study(direction='minimize')
study_cat.optimize(objective_cat, n_trials=20)

best_params_cat = study_cat.best_params
best_mse_cat    = study_cat.best_value

print(f"Best CatBoost MSE: {best_mse_cat:.4f}")
print(f"Best parameters:   {best_params_cat}")

Starting CatBoost tuning (20 trials, 3-fold CV)...
Best CatBoost MSE: 81.1923
Best parameters:   {'iterations': 394, 'learning_rate': 0.07061089397689581, 'depth': 5, 'l2_leaf_reg': 8.745942804968674}


In [11]:
# ============================================================
# SECTION 7A: CATBOOST SUBMISSION (best single model)
# ============================================================

from catboost import CatBoostRegressor

# Train CatBoost on full data with best params
cat_final = CatBoostRegressor(**best_params_cat, random_state=42, verbose=0)
cat_final.fit(X, y)

# Predict on test
cat_test_preds = cat_final.predict(X_test)
cat_test_preds = np.clip(cat_test_preds, 0, 100)

# Build submission
test_original = pd.read_csv(DATASET_PATH + 'test_x.csv', encoding='utf-8-sig')
submission_cat = pd.DataFrame({
    'student_id': test_original['student_id'],
    'career_success_score': cat_test_preds
})
submission_cat.to_csv(OUTPUT_PATH + 'submission_v4_catboost.csv', index=False)

print("CatBoost submission created!")
print(submission_cat.head())
print(f"\nPrediction range: {cat_test_preds.min():.2f} to {cat_test_preds.max():.2f}")
print(f"Prediction mean:  {cat_test_preds.mean():.2f}")

CatBoost submission created!
   student_id  career_success_score
0  STU_010001             56.975168
1  STU_010002             74.340180
2  STU_010003             73.396162
3  STU_010004             97.242124
4  STU_010005             77.609743

Prediction range: 35.50 to 100.00
Prediction mean:  76.11


## v4 Model Tuning Results (Optuna, 3-fold CV)

| Model    | CV MSE | Best Params Summary                          |
|----------|--------|----------------------------------------------|
| XGBoost  | 81.90  | n_est=538, lr=0.020, depth=6                 |
| LightGBM | 81.73  | n_est=499, lr=0.041, depth=6, leaves=20      |
| CatBoost | 81.19  | iter=394, lr=0.071, depth=5 (best single)    |

### Single Model Submission
- CatBoost (best single model) → **Kaggle MSE 91.303747**
- Improvement over v3 (97.33): 6.03 points — biggest jump so far!
- Hyperparameter tuning had the largest single impact

### Progress So Far
| Version | Description           | Kaggle    |
|---------|----------------------|-----------|
| V1      | Baseline XGBoost     | 99.089303 |
| V2      | Feature Engineering  | 98.827308 |
| V3      | NLP Features         | 97.334411 |
| V4 (cat)| Tuned CatBoost       | 91.303747 |

### Next: Ensemble (XGBoost + LightGBM + CatBoost)